# Notebook 03 — Inference Analysis

## Research Questions

1. **Q1 (Estimasi):** Berapa probabilitas sebuah PR di-merge, dan seberapa tidak pasti estimasi tersebut?
2. **Q2 (Inferensi):** Apakah rata-rata jumlah komentar berbeda secara signifikan antara PR yang merged vs unmerged?
3. **Q3 (Simulasi):** Berapa probabilitas sebuah issue butuh lebih dari 30 hari untuk ditutup?

## Member

* **Nama:** Luqman Harits Abdul Aziz
* **Role:** Inference Analyst (Member C)
* **Repository:** pandas-dev/pandas

## AI Usage Disclosure

**Member:** Luqman Harits Abdul Aziz — Inference Analyst (Member C)
**Tools used:** Gemini

| Task | Tool | Prompt summary | Output modified? |
| :--- | :--- | :--- | :--- |
| Menerjemahkan formula statistik ke kode Python | Gemini | "Bantu buatkan fungsi CI Poisson dan CI umum berdasarkan materi salindia" | Ya — disesuaikan dengan kerangka `inference.py` |
| *Troubleshooting* error integrasi *version control* | Gemini | "Kenapa hasil kode git checkout muncul fatal not a git repository" | Ya — disesuaikan dengan struktur folder *project* lokal |

In [7]:
import sys
import os

# Menyuruh Python mundur satu folder ke atas agar bisa membaca folder 'src'
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from src.inference import ci_poisson

# Load data PR bersih
df_pr = pd.read_csv('../data/clean/pr_clean.csv') 

print(df_pr.head())

# Pisahkan data jumlah komentar berdasarkan status PR
komentar_merged = df_pr[df_pr['status'] == 'merged']['comments'].dropna().tolist()

# Untuk PR yang unmerged
komentar_unmerged = df_pr[df_pr['status'] == 'unmerged']['comments'].dropna().tolist()

print(f"Jumlah sampel PR 'merged': {len(komentar_merged)}")
print(f"Jumlah sampel PR 'unmerged': {len(komentar_unmerged)}")

   number                                              title  status  \
0   65697  Revert "BUG: reject unhashable elements in Ind...  merged   
1   65693  [backport 3.0.x] TST: Adjust xfails for fastpa...  merged   
2   65686  BUG: Fix Index.where raising AssertionError wh...  merged   
3   65681              ASV: garbage collect as part of setup  merged   
4   65680    Bump github/codeql-action from 4.35.4 to 4.35.5  merged   

                  created_at                  closed_at  comments  \
0  2026-05-20 16:38:19+00:00  2026-05-20 21:22:12+00:00         1   
1  2026-05-20 14:03:23+00:00  2026-05-20 16:33:10+00:00         0   
2  2026-05-19 18:54:58+00:00  2026-05-21 14:34:32+00:00         7   
3  2026-05-18 21:24:09+00:00  2026-05-23 17:26:56+00:00         1   
4  2026-05-18 13:54:43+00:00  2026-05-18 15:27:45+00:00         0   

                 user  
0  jorisvandenbossche  
1  jorisvandenbossche  
2            anzinmhd  
3          rhshadrach  
4     dependabot[bot]  
Jumlah s

In [8]:
# Hitung Confidence Interval 95% menggunakan fungsi Poisson
ci_merged = ci_poisson(komentar_merged, confidence=0.95)
ci_unmerged = ci_poisson(komentar_unmerged, confidence=0.95)

print(f"Estimasi Rata-rata Komentar (Merged): {np.mean(komentar_merged):.2f}")
print(f"CI 95% untuk 'Merged': {ci_merged[0]:.2f} sampai {ci_merged[1]:.2f}\n")

print(f"Estimasi Rata-rata Komentar (Unmerged): {np.mean(komentar_unmerged):.2f}")
print(f"CI 95% untuk 'Unmerged': {ci_unmerged[0]:.2f} sampai {ci_unmerged[1]:.2f}")

Estimasi Rata-rata Komentar (Merged): 1.24
CI 95% untuk 'Merged': 1.14 sampai 1.34

Estimasi Rata-rata Komentar (Unmerged): 2.12
CI 95% untuk 'Unmerged': 1.96 sampai 2.29


## Kesimpulan dan Interpretasi

[cite_start]Berdasarkan hasil inferensi menggunakan *Confidence Interval* 95%[cite: 85, 86], kita dapat menjawab pertanyaan penelitian (Q2):
* Rata-rata jumlah komentar pada PR yang berstatus `merged` diproyeksikan berada pada interval **[1.14 sampai 1.34]**.
* Rata-rata jumlah komentar pada PR yang berstatus `unmerged` diproyeksikan berada pada interval **[1.96 sampai 2.29]**.

Karena kedua interval tersebut **[tidak saling tumpang tindih]**, maka dapat disimpulkan bahwa **[terdapat]** perbedaan yang signifikan secara statistik pada antusiasme/diskusi (jumlah komentar) antara PR yang diterima (merged) dan ditolak/ditutup (unmerged).